# M3L4 E05 — Router v1 vs Router v2
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

| Módulo | Concepto | Por qué lo necesitas acá |
|---|---|---|
| M3L4 E04 | `evaluate_router()`, golden dataset | Acá reusamos exactamente la misma función de evaluación |
| M3L4 E02 | `route_query()` con palabras clave | Vamos a comparar dos versiones del router: v1 (básica) vs v2 (mejorada) |
| M3L4 E03 | Misclassification como patrón de falla | Queremos medir si v2 reduce las misclassifications |
| Python | `pd.merge()`, DataFrames | Para comparar resultados de v1 y v2 lado a lado |

Si no completaste E04, hace eso primero. Acá comparamos routers, no construimos la evaluación desde cero.

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Router v1** | Versión básica con pocas palabras clave por dominio | `route_query_v1()` con ~1 keyword por intent |
| **Router v2** | Versión mejorada con más keywords + detección multi-intent + clarification | `route_query_v2()` con listas extensas de keywords |
| **Multi-intent** | Consulta que menciona temas de dos dominios distintos | v2 devuelve `'multi_intent'` cuando detecta >1 dominio |
| **Benchmark** | Prueba estandarizada para comparar versiones | Golden dataset de 12 casos evaluado con ambos routers |
| **Mejora medida** | Diferencia de accuracy entre v2 y v1 | `acc_v2 - acc_v1` (debe ser positiva si v2 es mejor) |

---

## Cómo encaja esto en un sistema de agentes

```
E04: Golden dataset + evaluate_router()
    |  Medir accuracy de UN router
    v
E05: Comparar DOS versiones del router (ESTE EJERCICIO)
    |  Router v1 (básico) vs Router v2 (mejorado)
    |  Misma métrica, mismo dataset
    v
Resultado: mejora cuantitativa demostrable
    |  "v2 mejoró de 67% a 92%"
    v
E06: Evaluar calidad de respuestas, no solo routing
    |  Pasamos de "llegó al agente correcto" a "respondió bien"
```

**Objetivo del ejercicio:** demostrar cómo validar mejoras de un router usando el golden dataset como benchmark.

## Instalación e imports

| Import | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `pandas` (via pip) | DataFrames para análisis y comparación de resultados | Para evaluar, mostrar y comparar ambos routers |
| `pd.merge()` | Combinar dos DataFrames por columna común | Para comparar `actual_intent` de v1 vs v2 caso por caso |

```python
!pip install pandas -q
import pandas as pd
```

In [ ]:
!pip install pandas -q
import pandas as pd
print('pandas listo.')

## Router v1 — Reglas básicas

Versión inicial con reglas simples: una palabra clave por dominio.

| Condición | Intent |
|---|---|
| `'vacaciones' in q` | `hr` |
| `'vpn' in q or 'error' in q` | `it` |
| `'factura' in q` | `finance` |
| `'contrato' in q` | `legal` |
| Si no | `general` |

**Limitaciones:** no detecta sinónimos ("licencia", "nómina"), no maneja queries cortas, no detecta multi-intent.

In [ ]:
def route_query_v1(query: str) -> str:
    q = query.lower()
    if 'vacaciones' in q:
        return 'hr'
    if 'vpn' in q or 'error' in q:
        return 'it'
    if 'factura' in q:
        return 'finance'
    if 'contrato' in q:
        return 'legal'
    return 'general'

print('Router v1 listo.')

## Router v2 — Más keywords + detección multi-intent

El v2 mejora el v1 de tres maneras:

1. **Más keywords por dominio:** cubre variaciones del vocabulario real (ej: no solo "vacaciones" sino también "licencia", "recibo", "beneficios", "portal rrhh")
2. **Detección de multi-intent:** cuando hay keywords de dos dominios, devuelve `'multi_intent'` en vez de elegir uno al azar
3. **Detección de clarification:** queries muy cortas (<= 2 palabras) van a `'clarification'`

```python
hr_keywords = ['vacaciones', 'licencia', 'recibo', 'nómina', 'beneficios', 'rrhh', 'portal rrhh']
it_keywords = ['vpn', 'error', 'app', 'laptop', 'wifi', 'login', 'contraseña', 'sistema', 'acceso']
finance_keywords = ['factura', 'pago', 'reembolso', 'gasto', 'cobro', 'comprobante', 'salario']
legal_keywords = ['contrato', 'legal', 'confidencialidad', 'nda', 'acuerdo', 'firma']
```

Por qué importa: un router con más keywords reduce los falsos negativos (consultas que deberían clasificarse pero caen en `general`).

In [ ]:
def route_query_v2(query: str) -> str:
    q = query.lower()
    hr_keywords      = ['vacaciones', 'licencia', 'recibo', 'nómina', 'beneficios', 'rrhh', 'portal rrhh']
    it_keywords      = ['vpn', 'error', 'app', 'laptop', 'wifi', 'login', 'contraseña', 'sistema', 'acceso']
    finance_keywords = ['factura', 'pago', 'reembolso', 'gasto', 'cobro', 'comprobante', 'salario']
    legal_keywords   = ['contrato', 'legal', 'confidencialidad', 'nda', 'acuerdo', 'firma']

    detected = []
    if any(w in q for w in hr_keywords):      detected.append('hr')
    if any(w in q for w in it_keywords):      detected.append('it')
    if any(w in q for w in finance_keywords): detected.append('finance')
    if any(w in q for w in legal_keywords):   detected.append('legal')

    if len(detected) > 1:  return 'multi_intent'
    if len(detected) == 1: return detected[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

print('Router v2 listo.')

## Golden Dataset — Extendido

Dataset de 12 casos que cubre todos los intents incluyendo casos frontera (``multi-intent``, ``clarification``). Se agregaron 4 casos nuevos respecto a E04 para probar las mejoras de v2.

In [ ]:
golden_dataset = [
    {'id': 'case_001', 'query': '¿Cómo solicito mis días de vacaciones?',             'expected_intent': 'hr'},
    {'id': 'case_002', 'query': 'Mi VPN no conecta desde ayer',                        'expected_intent': 'it'},
    {'id': 'case_003', 'query': 'Necesito ver mi factura del mes pasado',               'expected_intent': 'finance'},
    {'id': 'case_004', 'query': 'Necesito el contrato de confidencialidad actualizado', 'expected_intent': 'legal'},
    {'id': 'case_005', 'query': 'No puedo entrar al portal para ver mi recibo',        'expected_intent': 'hr'},
    {'id': 'case_006', 'query': 'ayuda',                                                'expected_intent': 'clarification'},
    {'id': 'case_007', 'query': '¿Cuándo se procesa el reembolso de gastos?',          'expected_intent': 'finance'},
    {'id': 'case_008', 'query': 'El sistema de login no me deja entrar',                'expected_intent': 'it'},
    {'id': 'case_009', 'query': 'Quiero pedir una licencia por enfermedad',             'expected_intent': 'hr'},
    {'id': 'case_010', 'query': '¿Puedo ver el comprobante de pago de mi salario?',    'expected_intent': 'finance'},
    {'id': 'case_011', 'query': 'Necesito firmar un NDA con un proveedor externo',     'expected_intent': 'legal'},
    {'id': 'case_012', 'query': 'ok',                                                   'expected_intent': 'clarification'},
]

print(f'Dataset: {len(golden_dataset)} casos')

## TODO — Función `evaluate_router` y comparación

Completa la función de evaluación (igual que en E04) y compara ambos routers.

### Flujo esperado

```python
df_v1, acc_v1 = evaluate_router(route_query_v1, golden_dataset)
df_v2, acc_v2 = evaluate_router(route_query_v2, golden_dataset)

print(f'V1: {acc_v1:.2%}')
print(f'V2: {acc_v2:.2%}')
print(f'Mejora: {(acc_v2 - acc_v1):.2%}')
```

> **Principio clave:** no se mejora un agente por intuición. Se mejora con comparación antes/después.

In [ ]:
def evaluate_router(router_fn, dataset: list) -> tuple:
    # TODO: mismo patrón que E04
    # Devuelve (df, accuracy)
    pass

print('Función definida.')

In [ ]:
# TODO: evaluar ambos routers
df_v1, acc_v1 = None, None  # reemplazar
df_v2, acc_v2 = None, None  # reemplazar

print(f'Router v1 accuracy: {acc_v1:.2%}')
print(f'Router v2 accuracy: {acc_v2:.2%}')

In [ ]:
# TODO: crear DataFrame de comparación con columnas [version, routing_accuracy]
comparison = None  # reemplazar
comparison

## Análisis de diferencias

Donde v1 y v2 difieren, podemos ver exactamente qué casos mejoró la nueva versión.

```python
merged = df_v1.merge(df_v2, on='id', suffixes=('_v1', '_v2'))
diffs = merged[merged['actual_intent_v1'] != merged['actual_intent_v2']]
```

Esto muestra los casos donde el cambio de reglas tuvo efecto.

In [ ]:
# TODO: mostrar los casos donde v1 y v2 difieren
# Pista: merge de df_v1 y df_v2 por 'id', comparar columnas actual_intent
pass

In [ ]:
assert acc_v1 is not None and acc_v2 is not None
assert acc_v2 >= acc_v1, f'v2 ({acc_v2:.2%}) debería ser >= v1 ({acc_v1:.2%})'
print(f'Checks E05 OK — Mejora: {(acc_v2 - acc_v1):.2%}')

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| No implementar `evaluate_router()` | La función queda como `pass` | `acc_v1` y `acc_v2` son `None` |
| v2 parece peor que v1 | El dataset no cubre los casos donde v2 mejora | Revisar si el dataset incluye sinónimos y multi-intent |
| No comparar caso por caso | Solo mirar accuracy global | No se ven los casos específicos que cambiaron |
| v2 no maneja `multi_intent` | No hay casos multi-intent en el dataset | Agregar `case_013` con keywords mezcladas |
| Confundir mejora con ruido | Diferencia de 1 caso (8%) se considera mejora significativa | Con datasets chicos, la métrica es sensible |

## Síntesis

### Qué construiste

| Componente | Descripción |
|---|---|
| `route_query_v1()` | Router básico con reglas mínimas |
| `route_query_v2()` | Router mejorado con keywords extendidas + multi-intent + clarification |
| `evaluate_router()` | Función reutilizable de E04 para medir accuracy |
| Comparación v1 vs v2 | Demostración cuantitativa de mejora |

### Flujo de mejora continua

```
Router v1 -> Medir accuracy (E04, E05)
    |
    v
Identificar fallos -> Agregar keywords, reglas, casos
    |
    v
Router v2 -> Medir accuracy otra vez
    |
    v
Mejora demostrada: v2 > v1
```

### Relación con otros ejercicios

| Ejercicio | Conexión con E05 |
|---|---|
| **E06** | Evaluator agent: evaluar calidad de respuestas, no solo routing |
| **E11** | Golden dataset scores: mantener el benchmark actualizado con cada versión |
| **E12** | Ciclo de mejora iterativa: medir -> mejorar -> medir de nuevo |